In [5]:
import numpy as np
import pandas as pd
import os
import warnings
import matplotlib.pyplot as plt
from arch import arch_model
from arch.univariate import ARX
import scipy
from scipy.stats import norm
from sklearn.mixture import GaussianMixture

### Import data

In [6]:
path = os.path.abspath('E:/RA/Geert/task1.py')
dir_path = os.path.dirname(path)
os.chdir(dir_path)
excel_file = pd.ExcelFile('Aggregate_CPI_inflation_20230513.xls')
sheet_quarter = excel_file.sheet_names[0]
sheet_month = excel_file.sheet_names[1]
#quarterly and monthly aggregate CPI data (deseasonalized). The full sample is 1947-2022
data_quarter = excel_file.parse(sheet_quarter, skiprows=2)
data_month = excel_file.parse(sheet_month, skiprows=2)
data_quarter.index = pd.to_datetime(data_quarter['Year'].astype(str) + '-Q' + data_quarter['Quarter'].astype(str))
data_month.index = pd.to_datetime(data_month[['Year', 'Month']].assign(day=1))
data_quarter.columns = ['Year', 'Quarter', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
data_month.columns = ['Year', 'Month', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
sample_data = data_quarter[data_quarter['Year']>1969]

In [8]:
sample_data['Inflation_lag_1'] =  sample_data['Inflation'].shift(1)
sample_data['Inflation_lag_2'] =  sample_data['Inflation'].shift(2)
sample_data['Forecasted_inflation_lag_1'] =  sample_data['Forecasted inflation'].shift(1)
sample_data = sample_data.dropna()

 ### Gaussian Mixture distribution

The mixture of 2 normal distribution given parameters $p_1$,$\mu_1$,$\sigma^2_1$ is 
$$
 f\left(z_t\right) =   
 p_1  \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(z_t-\mu_1)^2}{2\sigma_1^2} \}
+p_2\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(z_t-\mu_2)^2}{2\sigma_2^2} \}
$$
where $p_2 = 1- p_1 $,$\mu_2 = \frac{-p_1\mu_1}{p_2}$, and
$\sigma_2^2 =\frac{1-p_2\mu_2^2 -p_1(\sigma_1^2 + \mu_1^2 )}{p_2} $

N th order moment can be calculated as:
$$
E[z^n] = \int z^n   \left[p_1 \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(z-\mu_1)^2}{2\sigma_1^2} \}
+(1-p_1)\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(z-\mu_2)^2}{2\sigma_2^2} \}  \right] dz
$$

$$
E[z^n] =  p_1 E[x_1^n|x_1 \sim N(\mu_1,\sigma_1^2) ] + p_2 E[x_2^n|x_2 \sim N(\mu_2,\sigma_2^2) ]
$$

Similary, CDF can be calculated analytically by
$$
\Phi(a) = \int_{-\inf}^{a} z   \left[p_1 \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(z-\mu_1)^2}{2\sigma_1^2} \}
+p_2\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(z-\mu_2)^2}{2\sigma_2^2} \}  \right] dz
$$

$$
\Phi(a) = p_1\Phi_{x_1}(a) + p_2\Phi_{x_2}(a)
$$

We can think mixture of two normal distributions generated by three random variable $Z$,$X_1$,$X_2$. $Z$ follows Bernoulli($P_1$), $X_1 \sim  N(\mu_1,\sigma_1^2)$ and $X_2 \sim  N(\mu_2,\sigma_2^2)$. In order to generate a random sample point, we can first use $Z$ to decide which normal distributions to use, and then randomly pick a point in that normal distribution. 

Log likelihood function is
$$
 L\left(\epsilon_t\right) =  \sum_{t=1}^{T} \ln \frac{1}{\sqrt{h_t}} \left[
 p_1  \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(z_t-\mu_1)^2}{2\sigma_1^2} \}
+p_2\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(z_t-\mu_2)^2}{2\sigma_2^2} \} \right]
$$,
where $z_t = \frac{\epsilon_t}{\sqrt{h_t}}$

### Gridly search optimal starting points
1. Find some starting points with decent standard error and AIC
2. Gridly search optimal starting points within 1 standard error neighborhood of above points

In [10]:
constraints = {'type': 'ineq',
                'fun': lambda x:np.array([ 1 - x[-4] * (x[-3]**2 + x[-2]) - (1 - x[-4]) * (-x[-4] * x[-3] / (1 - x[-4]))**2 +x[-1] ])} 
options={'maxiter': 1000,'method':'Powell' ,'constraints':constraints}

#### Construct Mixture normal distribution 

$
\pi (t) =  fc(t-1) + \epsilon (t)
$

In [14]:
sparch11 = arch_model(y=Y, mean=mean, vol='GARCH',p=1, q=1,lags=lags)
sparch11.distribution = MixNormal()
gjr_MixNormal=sparch11.fit(disp='off',options=options)#,starting_values=starting_values )
gjr_MixNormal.summary()

C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_12016/2492726350.py:106: RuntimeWarning: divide by zero encountered in log
  lls =-1/2 *log(sigma2) +  log(   p1/(sqrt(2*pi*sigma_1_2)) *  exp(-( z-u1)**2/(2*sigma_1_2))
D:\anaconda\lib\site-packages\arch\univariate\base.py:759: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


<class 'statsmodels.iolib.summary.Summary'>
"""
                                Zero Mean - GARCH Model Results                                
===============================================================================================
Dep. Variable:                         Inflation shock   R-squared:                       0.000
Mean Model:                                  Zero Mean   Adj. R-squared:                  0.005
Vol Model:                                       GARCH   Log-Likelihood:               -199.978
Distribution:      Mixture of two Normal distributions   AIC:                           413.957
Method:                             Maximum Likelihood   BIC:                           437.319
                                                         No. Observations:                  208
Date:                                 Thu, Jul 13 2023   Df Residuals:                      208
Time:                                         11:41:38   Df Model:                            0
                            Volatility Model                            
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
omega          4.6548      2.936      1.585      0.113 [ -1.100, 10.409]
alpha[1]       0.1525      1.302      0.117      0.907 [ -2.400,  2.705]
beta[1]        0.8475  3.260e-02     26.000 4.933e-149 [  0.784,  0.911]
                                 Distribution                                
=============================================================================
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
p_1            0.0100  8.558e-03      1.169      0.243 [-6.773e-03,2.677e-02]
mu_1          -2.3795      2.700     -0.881      0.378      [ -7.672,  2.913]
sigma_1^2      0.8522      1.150      0.741      0.459      [ -1.401,  3.105]
sigma_2^2      0.0116  6.373e-03      1.824  6.815e-02 [-8.665e-04,2.412e-02]
=============================================================================

Covariance estimator: robust
WARNING: The optimizer did not indicate successful convergence. The message was Positive directional derivative for linesearch.
See convergence_flag.

"""

In [62]:
sparch11 = arch_model(y=Y, mean=mean, vol='GARCH',p=2, q=1,lags=lags)
sparch11.distribution = MixNormal()
gjr_MixNormal=sparch11.fit(disp='off',options=options)#,starting_values=starting_values )
gjr_MixNormal.summary()

C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/1117864732.py:107: RuntimeWarning: invalid value encountered in sqrt
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )


<class 'statsmodels.iolib.summary.Summary'>
"""
                                Zero Mean - GARCH Model Results                                
===============================================================================================
Dep. Variable:                         Inflation shock   R-squared:                       0.000
Mean Model:                                  Zero Mean   Adj. R-squared:                  0.005
Vol Model:                                       GARCH   Log-Likelihood:               -170.495
Distribution:      Mixture of two Normal distributions   AIC:                           354.990
Method:                             Maximum Likelihood   BIC:                           378.353
                                                         No. Observations:                  208
Date:                                 Wed, Jul 12 2023   Df Residuals:                      208
Time:                                         22:33:17   Df Model:                            0
                            Volatility Model                            
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
omega          0.2843      0.304      0.935      0.350 [ -0.312,  0.880]
alpha[1]       1.0000      0.939      1.065      0.287 [ -0.840,  2.840]
alpha[2]       0.9774      0.847      1.153      0.249 [ -0.683,  2.638]
beta[1]    1.0766e-12      0.120  9.004e-12      1.000 [ -0.234,  0.234]
                                Distribution                               
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
p_1            0.9222  5.636e-02     16.362  3.547e-60    [  0.812,  1.033]
mu_1           0.0574  3.499e-02      1.641      0.101 [-1.116e-02,  0.126]
sigma_1^2      0.3005      0.257      1.171      0.242    [ -0.203,  0.804]
===========================================================================

Covariance estimator: robust
"""

In [11]:
Y=sample_data['Inflation shock']
X=None
mean='Zero'
lags=None
cov = 'robust'
options = {'maxiter': 1000}
warnings.filterwarnings('ignore', category=scipy.optimize.OptimizeWarning)

In [10]:
gjr_normal = arch_model(y=Y,x=X, mean=mean, vol='GARCH',p=2, o=2,q=1,dist='normal',lags=lags).fit(disp='off',cov_type=cov,options=options)
stdresid = gjr_normal.resid / gjr_normal.conditional_volatility
gmm = GaussianMixture(n_components=2).fit(np.array(stdresid.dropna()).reshape(-1,1))
dparams = np.array([gmm.weights_[0],gmm.means_ [0][0], gmm.covariances_[0][0][0]]  )
starting_values = np.concatenate( (np.array(gjr_normal.params),dparams )   )
sparch21 = arch_model(y=Y,x=X, mean=mean, vol='GARCH',p=2, o=2, q=1,lags=lags)
sparch21.distribution = MixNormal()
gjr_MixNormal=sparch21.fit(disp='off',cov_type=cov,options=options)#,starting_values=starting_values )
gjr_MixNormal.summary()

C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_4900/2300885567.py:106: RuntimeWarning: divide by zero encountered in log
  lls =-1/2 *log(sigma2) +  log(   p1/(sqrt(2*pi*sigma_1_2)) *  exp(-( z-u1)**2/(2*sigma_1_2))
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_4900/2300885567.py:107: RuntimeWarning: invalid value encountered in sqrt
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_4900/2300885567.py:107: RuntimeWarning: overflow encountered in exp
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
D:\anaconda\lib\site-packages\arch\univariate\base.py:759: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


<class 'statsmodels.iolib.summary.Summary'>
"""
                              Zero Mean - GJR-GARCH Model Results                              
===============================================================================================
Dep. Variable:                         Inflation shock   R-squared:                       0.000
Mean Model:                                  Zero Mean   Adj. R-squared:                  0.005
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -211.746
Distribution:      Mixture of two Normal distributions   AIC:                           441.491
Method:                             Maximum Likelihood   BIC:                           471.615
                                                         No. Observations:                  210
Date:                                 Thu, Jul 13 2023   Df Residuals:                      210
Time:                                         11:10:38   Df Model:                            0
                             Volatility Model                             
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
omega          0.1481  6.740e-02      2.197  2.803e-02 [1.597e-02,  0.280]
alpha[1]       0.0000  5.855e-02      0.000      1.000   [ -0.115,  0.115]
alpha[2]       0.5142      0.720      0.714      0.475   [ -0.896,  1.925]
gamma[1]   1.5871e-07      0.135  1.177e-06      1.000   [ -0.264,  0.264]
gamma[2]      -0.5142      0.744     -0.691      0.489   [ -1.972,  0.944]
beta[1]        0.6627  5.494e-02     12.063  1.656e-33   [  0.555,  0.770]
                                Distribution                                
============================================================================
                 coef    std err          t      P>|t|      95.0% Conf. Int.
----------------------------------------------------------------------------
p_1            0.0100  1.354e-03      7.385  1.522e-13 [7.346e-03,1.265e-02]
mu_1          -2.9060     12.762     -0.228      0.820     [-27.919, 22.107]
sigma_1^2  1.0000e-03      6.266  1.596e-04      1.000     [-12.281, 12.283]
============================================================================

Covariance estimator: robust
WARNING: The optimizer did not indicate successful convergence. The message was Inequality constraints incompatible.
See convergence_flag.

"""

start with sparch. and iteration like boosting

In [6]:
def constraint(a: Float64Array, b: Float64Array) -> list[dict[str, object]]:
    """
    Generate constraints from arrays

    Parameters
    ----------
    a : ndarray
        Parameter loadings
    b : ndarray
        Constraint bounds

    Returns
    -------
    constraints : dict
        Dictionary of inequality constraints, one for each row of a

    Notes
    -----
    Parameter constraints satisfy a.dot(parameters) - b >= 0
    """

    def factory(coeff: Float64Array, val: float) -> Callable[..., float]:
        def f(params: Float64Array, *args: Any) -> float:
            return np.dot(coeff, params) - val

        return f

    constraints = []
    for i in range(a.shape[0]):
        con = {"type": "ineq", "fun": factory(a[i], b[i])}
        constraints.append(con)

    return constraints

In [7]:
a = array([[1, 0, 0], [-1, 0, 0], [0, 1,0], [0, -1,0] , [0,0,1],[0,0,-1]])
b = array([0.01, 0.99, -20,20,0.001, 10])
constraint(a, b)

[{'type': 'ineq',
  'fun': <function __main__.constraint.<locals>.factory.<locals>.f(params: 'Float64Array', *args: 'Any') -> 'float'>},
 {'type': 'ineq',
  'fun': <function __main__.constraint.<locals>.factory.<locals>.f(params: 'Float64Array', *args: 'Any') -> 'float'>},
 {'type': 'ineq',
  'fun': <function __main__.constraint.<locals>.factory.<locals>.f(params: 'Float64Array', *args: 'Any') -> 'float'>},
 {'type': 'ineq',
  'fun': <function __main__.constraint.<locals>.factory.<locals>.f(params: 'Float64Array', *args: 'Any') -> 'float'>},
 {'type': 'ineq',
  'fun': <function __main__.constraint.<locals>.factory.<locals>.f(params: 'Float64Array', *args: 'Any') -> 'float'>},
 {'type': 'ineq',
  'fun': <function __main__.constraint.<locals>.factory.<locals>.f(params: 'Float64Array', *args: 'Any') -> 'float'>}]

In [47]:
best_aic = 400
best_summary = None
for i in range(500):
    starting_values = np.concatenate( (np.array(gjr_normal.params),dparams )   )
    sparch21 = arch_model(y=Y,x=X, mean=mean, vol='GARCH',p=2, o=2, q=1,lags=lags)
    sparch21.distribution = MixNormal()
    '''    try:
        gjr_MixNormal=sparch21.fit(disp='off',cov_type=cov,options=options,starting_values=starting_values )
          # If the fitting is successful, break the loop
    except:'''
    gjr_MixNormal=sparch21.fit(disp='off',cov_type=cov,options=options,starting_values=starting_values )
    if best_aic > gjr_MixNormal.aic:
        best_params = np.array(gjr_MixNormal.params)
        best_aic = gjr_MixNormal.aic
        best_summary = gjr_MixNormal.summary()
        print(best_summary)
    dparams =  np.array(  [ np.random.uniform(),np.random.uniform()*10-5, np.random.uniform()] )
                       
        
best_summary

D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: invalid value encountered in sqrt
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: overflow encountered in exp
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:106: RuntimeWarning: divide by zero encountered in log
  lls =-1/2 *log(sigma2) +  log(   p1/(sqrt(2*pi*sigma_1_2)) *  exp(-( z-u1)**2/(2*sigma_1_2))
D:\anaconda\lib\site-packages\arch\univariate\base.py:759: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional

                              Zero Mean - GJR-GARCH Model Results                              
Dep. Variable:                         Inflation shock   R-squared:                       0.000
Mean Model:                                  Zero Mean   Adj. R-squared:                  0.005
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -171.860
Distribution:      Mixture of two Normal distributions   AIC:                           361.719
Method:                             Maximum Likelihood   BIC:                           391.757
                                                         No. Observations:                  208
Date:                                 Wed, Jul 12 2023   Df Residuals:                      208
Time:                                         22:23:57   Df Model:                            0
                              Volatility Model                             
                 coef    std err          t      P>|t|     9

D:\anaconda\lib\site-packages\arch\univariate\base.py:759: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: invalid value encountered in sqrt
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: overflow encountered in exp
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:106: RuntimeWarning: divide by zero encountered in log
  

D:\anaconda\lib\site-packages\arch\univariate\base.py:759: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: invalid value encountered in sqrt
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: overflow encountered in exp
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:106: RuntimeWarning: divide by zero encountered in log
  

D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: invalid value encountered in sqrt
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: overflow encountered in exp
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:106: RuntimeWarning: divide by zero encountered in log
  lls =-1/2 *log(sigma2) +  log(   p1/(sqrt(2*pi*sigma_1_2)) *  exp(-( z-u1)**2/(2*sigma_1_2))
D:\anaconda\lib\site-packages\arch\univariate\base.py:759: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional

...

D:\anaconda\lib\site-packages\arch\univariate\base.py:759: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: invalid value encountered in sqrt
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: overflow encountered in exp
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:106: RuntimeWarning: divide by zero encountered in log
  

D:\anaconda\lib\site-packages\arch\univariate\base.py:759: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: invalid value encountered in sqrt
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: overflow encountered in exp
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:106: RuntimeWarning: divide by zero encountered in log
  

D:\anaconda\lib\site-packages\arch\univariate\base.py:759: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: invalid value encountered in sqrt
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: overflow encountered in exp
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:106: RuntimeWarning: divide by zero encountered in log
  

D:\anaconda\lib\site-packages\arch\univariate\base.py:759: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: invalid value encountered in sqrt
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:107: RuntimeWarning: overflow encountered in exp
  +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_15172/314617582.py:106: RuntimeWarning: divide by zero encountered in log
  

<class 'statsmodels.iolib.summary.Summary'>
"""
                              Zero Mean - GJR-GARCH Model Results                              
===============================================================================================
Dep. Variable:                         Inflation shock   R-squared:                       0.000
Mean Model:                                  Zero Mean   Adj. R-squared:                  0.005
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -171.860
Distribution:      Mixture of two Normal distributions   AIC:                           361.719
Method:                             Maximum Likelihood   BIC:                           391.757
                                                         No. Observations:                  208
Date:                                 Wed, Jul 12 2023   Df Residuals:                      208
Time:                                         22:23:57   Df Model:                            0
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega          0.1271  4.431e-02      2.868  4.133e-03  [4.023e-02,  0.214]
alpha[1]       0.2339      0.154      1.523      0.128 [-6.702e-02,  0.535]
alpha[2]       0.7756      0.345      2.251  2.438e-02    [  0.100,  1.451]
gamma[1]      -0.0973      0.166     -0.587      0.557    [ -0.422,  0.228]
gamma[2]      -0.7756      0.338     -2.297  2.165e-02    [ -1.438, -0.114]
beta[1]        0.1036      0.125      0.831      0.406    [ -0.141,  0.348]
                                 Distribution                                
=============================================================================
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
p_1        1.0073e-03  6.244e-04      1.613      0.107 [-2.165e-04,2.231e-03]
mu_1        -100.0000     69.601     -1.437      0.151   [-2.364e+02, 36.415]
sigma_1^2  1.0000e-03    129.407  7.728e-06      1.000 [-2.536e+02,2.536e+02]
=============================================================================

Covariance estimator: robust
WARNING: The optimizer did not indicate successful convergence. The message was Positive directional derivative for linesearch.
See convergence_flag.

"""

In [88]:
best_aic

400

In [82]:
gjr_normal.aic

361.4027665224535

In [76]:
sparch = arch_model(y=sample_data['Inflation shock'],mean='Zero', vol='GARCH',p=2,o=2, q=1)
sparch.distribution = MixNormal()
res = sparch.fit(cov_type='robust')
res.summary()

Iteration:      1,   Func. Count:     11,   Neg. LLF: nan
Iteration:      2,   Func. Count:     24,   Neg. LLF: 327.96910822050086
Iteration:      3,   Func. Count:     35,   Neg. LLF: inf
Iteration:      4,   Func. Count:     46,   Neg. LLF: 443.8742989412418
Iteration:      5,   Func. Count:     57,   Neg. LLF: 304.50505794201734
Iteration:      6,   Func. Count:     67,   Neg. LLF: 248.98484052655522
Iteration:      7,   Func. Count:     77,   Neg. LLF: 338.6645365784585
Iteration:      8,   Func. Count:     89,   Neg. LLF: 236.74404916964983
Iteration:      9,   Func. Count:     99,   Neg. LLF: 223.49876359093895
Iteration:     10,   Func. Count:    109,   Neg. LLF: 221.65687442029113
Iteration:     11,   Func. Count:    119,   Neg. LLF: 349.9214662571972
Iteration:     12,   Func. Count:    132,   Neg. LLF: inf
Iteration:     13,   Func. Count:    144,   Neg. LLF: 222.12759866107046
Iteration:     14,   Func. Count:    155,   Neg. LLF: 223.681193196177
Iteration:     15,   Func. C

C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_10968/1382309230.py:102: RuntimeWarning: invalid value encountered in sqrt
  sigma_2 = sqrt(p1/(p1-1)*sigma_1**2 - p1/(p1-1)**2 * u1**2 + 1/(1-p1))
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_10968/1382309230.py:105: RuntimeWarning: divide by zero encountered in log
  lls =-1/2 *log(sigma2) +  log(   p1/(sqrt(2*pi)*sigma_1) *  exp(-( (z-u1)/sqrt(sigma2))**2/(2*sigma_1**2))
D:\anaconda\lib\site-packages\arch\univariate\base.py:756: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


<class 'statsmodels.iolib.summary.Summary'>
"""
                              Zero Mean - GJR-GARCH Model Results                              
===============================================================================================
Dep. Variable:                         Inflation shock   R-squared:                       0.000
Mean Model:                                  Zero Mean   Adj. R-squared:                  0.005
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -170.277
Distribution:      Mixture of two Normal distributions   AIC:                           358.554
Method:                             Maximum Likelihood   BIC:                           388.592
                                                         No. Observations:                  208
Date:                                 Mon, Jul 10 2023   Df Residuals:                      208
Time:                                         18:28:29   Df Model:                            0
                            Volatility Model                            
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
omega          1.0111      0.955      1.059      0.290 [ -0.861,  2.883]
alpha[1]       0.4314      0.295      1.462      0.144 [ -0.147,  1.010]
alpha[2]       0.7558      0.654      1.156      0.248 [ -0.526,  2.037]
gamma[1]      -0.4314      0.395     -1.093      0.274 [ -1.205,  0.342]
gamma[2]      -0.6530      0.693     -0.943      0.346 [ -2.011,  0.705]
beta[1]        0.0942      0.763      0.123      0.902 [ -1.401,  1.589]
                                 Distribution                                
=============================================================================
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
p_1        1.1988e-03  2.476e-05     48.424      0.000  [1.150e-03,1.247e-03]
mu_1         -22.8118      0.467    -48.887      0.000      [-23.726,-21.897]
sigma_1    1.0000e-04  1.613e-04      0.620      0.535 [-2.162e-04,4.162e-04]
=============================================================================

Covariance estimator: robust
WARNING: The optimizer did not indicate successful convergence. The message was Positive directional derivative for linesearch.
See convergence_flag.

"""

In [97]:
p1, u1, sigma_1 = res.params[-3:]
p2 = 1- p1
u2 = p1/(p1-1) *u1
sigma_2 = sqrt(p1/(p1-1)*sigma_1**2 - p1/(p1-1)**2 * u1**2 + 1/(1-p1))
print(p2,u2,sigma_2)

0.5862508004710025 0.00040192261105125 1.2237756702011555


GJR with standardized student's t distribution

In [90]:
garch = arch_model(y=sample_data['Inflation shock'],mean='Zero', vol='GARCH',p=2,o=2, q=1,dist='studentst').fit()
garch.summary()

Iteration:      1,   Func. Count:      9,   Neg. LLF: 1059.2480705903863
Iteration:      2,   Func. Count:     18,   Neg. LLF: 361.4208285990386
Iteration:      3,   Func. Count:     28,   Neg. LLF: 182.3912077139476
Iteration:      4,   Func. Count:     37,   Neg. LLF: 176.1067498280596
Iteration:      5,   Func. Count:     46,   Neg. LLF: 172.9383585441007
Iteration:      6,   Func. Count:     55,   Neg. LLF: 177.73311238921468
Iteration:      7,   Func. Count:     64,   Neg. LLF: 168.76468750438636
Iteration:      8,   Func. Count:     73,   Neg. LLF: 170.00896267372696
Iteration:      9,   Func. Count:     82,   Neg. LLF: 168.00839523498007
Iteration:     10,   Func. Count:     91,   Neg. LLF: 166.8306511724226
Iteration:     11,   Func. Count:     99,   Neg. LLF: 166.82095617680724
Iteration:     12,   Func. Count:    107,   Neg. LLF: 166.81543313417757
Iteration:     13,   Func. Count:    115,   Neg. LLF: 166.80529252677775
Iteration:     14,   Func. Count:    123,   Neg. LLF: 16

<class 'statsmodels.iolib.summary.Summary'>
"""
                        Zero Mean - GJR-GARCH Model Results                         
====================================================================================
Dep. Variable:              Inflation shock   R-squared:                       0.000
Mean Model:                       Zero Mean   Adj. R-squared:                  0.005
Vol Model:                        GJR-GARCH   Log-Likelihood:               -166.805
Distribution:      Standardized Student's t   AIC:                           347.610
Method:                  Maximum Likelihood   BIC:                           370.973
                                              No. Observations:                  208
Date:                      Tue, Jun 20 2023   Df Residuals:                      208
Time:                              23:34:02   Df Model:                            0
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega          0.1034  3.329e-02      3.106  1.893e-03  [3.817e-02,  0.169]
alpha[1]       0.4484      0.194      2.308  2.102e-02  [6.757e-02,  0.829]
alpha[2]       0.7420      0.298      2.491  1.275e-02    [  0.158,  1.326]
gamma[1]      -0.1674      0.235     -0.713      0.476    [ -0.628,  0.293]
gamma[2]      -0.7420      0.288     -2.576  9.993e-03    [ -1.307, -0.177]
beta[1]        0.0980  9.150e-02      1.071      0.284 [-8.133e-02,  0.277]
                              Distribution                              
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
nu             5.4054      2.004      2.697  6.996e-03 [  1.477,  9.334]
========================================================================

Covariance estimator: robust
"""

In [13]:
from __future__ import annotations
from arch.univariate.distribution import Distribution
from abc import ABCMeta, abstractmethod
from collections.abc import Sequence
from typing import Callable
import warnings

from numpy import (
    abs,    array,    asarray,    empty,    exp,    int64,    
    integer,    isscalar,    log,    nan,    ndarray, exp,
    ones_like,    pi,    sign,    sqrt,    sum,)
from numpy.random import Generator, RandomState, default_rng
from scipy.special import comb, gamma, gammainc, gammaincc, gammaln
import scipy.stats as stats
from scipy.optimize import bisect

from arch.typing import ArrayLike, ArrayLike1D, Float64Array
from arch.utility.array import AbstractDocStringInheritor, ensure1d

class MixNormal(Distribution, metaclass=AbstractDocStringInheritor):
    """
    Mixture of two Normal distributions for use with SPARCH model

    Parameters
    ----------
    random_state : RandomState, optional
        .. deprecated:: 5.0

           random_state is deprecated. Use seed instead.

    seed : {int, Generator, RandomState}, optional
        Random number generator instance or int to use. Set to ensure
        reproducibility. If using an int, the argument is passed to
        ``np.random.default_rng``.  If not provided, ``default_rng``
        is used with system-provided entropy.
    """

    def __init__(
        self,
        random_state: RandomState | None = None,
        *,
        seed: None | int | RandomState | Generator = None,
    ) -> None:
        super().__init__(random_state=random_state, seed=seed)
        self._name = "Mixture of two Normal distributions"
        self.num_params: int =4  

    def constraints(self) -> tuple[Float64Array, Float64Array]:
        return array([[1, 0, 0,0], [-1, 0, 0,0], [0, 1,0,0], [0, -1,0,0] , [0,0,1,0],[0,0,-1,0], [0,0,0,1],[0,0,0,-1]]), array([0.01, 0.99, -20,20,0.001, 10, 0.001, 10])

    def bounds(self, resids: Float64Array) -> list[tuple[float, float]]:
        """
        Bounds of parameters:
        p1: (0,1)
        u1: (-10,10)
        sigma_1:(0.001,10)
        sigma_2:(0.001,10)
        """
        return [(0.01, 0.99),(-20,20),(0.001, 10),(0.001, 10)]

    def loglikelihood(
        self,
        parameters: Sequence[float] | ArrayLike1D,
        resids: ArrayLike,
        sigma2: ArrayLike,
        individual: bool = False,
    ) -> float | Float64Array:
        r"""Computes the log-likelihood of assuming residuals are mixture normally
        distributed, conditional on the variance

        Parameters
        ----------
        parameters : ndarray
            Parameters of the first normal distribution: p1,u1,sigma1. Second one can be calculated by restrictions.
        resids  : ndarray
            The residuals to use in the log-likelihood calculation
        sigma2 : ndarray
            Conditional variances of resids
        individual : bool, optional
            Flag indicating whether to return the vector of individual log
            likelihoods (True) or the sum (False)

        Returns
        -------
        ll : float
            The log-likelihood

        Notes
        -----
        The log-likelihood of a single data point x is

        .. math::

            \ln f\left(x\right)=
            \ln \frac{1}{\sqrt{h_t}} \left[
                 p_1  \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(x-\mu_1)^2}{2\sigma_1^2} \}
                +(1-p_1)\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(x-\mu_2)^2}{2\sigma_2^2} \} \right]

        """
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2,sigma_2_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        
        z = resids/sqrt(sigma2)
        lls =-1/2 *log(sigma2) +  log(   p1/(sqrt(2*pi*sigma_1_2)) *  exp(-( z-u1)**2/(2*sigma_1_2)) 
                                     +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )      
        if individual:
            return lls
        else:
            return sum(lls)

    def starting_values(self, std_resid: Float64Array) -> Float64Array:
        """
        Starting values of parameters
        """
        gmm = GaussianMixture(n_components=2).fit(std_resid.reshape(-1,1))
        return array([gmm.weights_[0],gmm.means_ [0][0], gmm.covariances_[0][0][0],  gmm.covariances_[1][0][0]])
        #return array([0.8,0,0.6])

    
    def _simulator(self, size: int | tuple[int, ...]) -> Float64Array:
        assert self._parameters is not None
        p1, u1, sigma_1_2,sigma_2_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        
        Z = random.choice([0, 1], p=[1 - p1, p1])
        return sqrt(sigma_1_2)**Z*sqrt(sigma_2_2)**(1-Z)*self._generator.standard_normal(size) + u1*Z+ u2*(1-Z)

    def simulate(
        self, parameters: int | float | Sequence[float | int] | ArrayLike1D
    ) -> Callable[[int | tuple[int, ...]], Float64Array]:
        parameters = ensure1d(parameters, "parameters", False)
        self._parameters = asarray(parameters, dtype=float)
        return self._simulator

    def parameter_names(self) -> list[str]:
        return ['p_1','mu_1','sigma_1^2','sigma_2^2']

    def cdf(
        self,
        resids: Sequence[float] | ArrayLike1D,
        parameters: None | Sequence[float] | ArrayLike1D = None,
    ) -> Float64Array:
        self._check_constraints(parameters)
        
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2,sigma_2_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        
        return p1*stats.norm.cdf(asarray((resids-u1)/sqrt(sigma_1_2))  ) + p2*stats.norm.cdf(asarray((resids-u2)/sqrt(sigma_2_2)))

    def ppf(
        self,
        pits: float | Sequence[float] | ArrayLike1D,
        parameters: None | Sequence[float] | ArrayLike1D = None,
    ) -> Float64Array:
        self._check_constraints(parameters)
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2,sigma_2_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        
        scalar = isscalar(pits)
        if scalar:
            pits = array([pits])
        else:
            pits = asarray(pits)
            
        def inverse_cdf(cdf, target_p, lower_bound=-100, upper_bound=100):
            def root_func(x):
                return cdf(x,parameters) - target_p
            return bisect(root_func, lower_bound, upper_bound)     
        
        ppf = inverse_cdf(self.cdf, pits, lower_bound=-100, upper_bound=100)

        if scalar:
            return ppf[0]
        else:
            return ppf
        
   

    def moment(
        self, n: int, parameters: None | Sequence[float] | ArrayLike1D = None
    ) -> float:
        if n < 0:
            return nan
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2,sigma_2_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        
        moment1 = stats.norm.moment(n,loc=u1,scale=sqrt(sigma_1_2))
        moment2 = stats.norm.moment(n,loc=u2,scale=sqrt(sigma_2_2))
        return p1 * moment1 + p2 * moment2

    def partial_moment(
        self,
        n: int,
        z: float = 0.0,
        parameters: None | Sequence[float] | ArrayLike1D = None,
        num_samples=100000,
    ) -> float:
        
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2,sigma_2_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        
        if n < 0:
            return nan
        elif n == 0:
            return cdf(z,parameters)
        elif n==1:
            return -p1*stats.norm.pdf(z,loc=u1,scale=sqrt(sigma_1_2))  -p2*stats.norm.pdf(z,loc=u2,scale=sqrt(sigma_2_2))
        else:
            -(z ** (n - 1)) * (p1*stats.norm.pdf(z,loc=u1,scale=sqrt(sigma_1_2))+p2*stats.norm.pdf(z,loc=u2,scale=sqrt(sigma_2_2))) 
            + (n - 1) * self.partial_moment(  n - 2, z, parameters  )

In [ ]:
from arch.univariate.distribution import Distribution
from abc import ABCMeta, abstractmethod
from collections.abc import Sequence
from typing import Callable
import warnings

from numpy import (
    abs,    array,    asarray,    empty,    exp,    int64,    
    integer,    isscalar,    log,    nan,    ndarray, exp,
    ones_like,    pi,    sign,    sqrt,    sum, random)
from numpy.random import Generator, RandomState, default_rng
from scipy.special import comb, gamma, gammainc, gammaincc, gammaln
import scipy.stats as stats
from scipy.optimize import bisect

from arch.typing import ArrayLike, ArrayLike1D, Float64Array
from arch.utility.array import AbstractDocStringInheritor, ensure1d

class MixNormal(Distribution, metaclass=AbstractDocStringInheritor):
    """
    Mixture of two Normal distributions for use with SPARCH model

    Parameters
    ----------
    random_state : RandomState, optional
        .. deprecated:: 5.0

           random_state is deprecated. Use seed instead.

    seed : {int, Generator, RandomState}, optional
        Random number generator instance or int to use. Set to ensure
        reproducibility. If using an int, the argument is passed to
        ``np.random.default_rng``.  If not provided, ``default_rng``
        is used with system-provided entropy.
    """

    def __init__(
        self,
        random_state: RandomState | None = None,
        *,
        seed: None | int | RandomState | Generator = None,
    ) -> None:
        super().__init__(random_state=random_state, seed=seed)
        self._name = "Mixture of two Normal distributions"
        self.num_params: int = 5  

    def constraints(self) -> tuple[Float64Array, Float64Array]:
        return empty(0), empty(0)
        
    def bounds(self, resids: Float64Array) -> list[tuple[float, float]]:
        """
        Bounds of parameters:
        """
        return [(1e-10, 1 - 1e-10),(-10000, 10000), (-10000, 10000), (1e-10, 10000), (1e-10, 10000) ]
    

    def loglikelihood(
        self,
        parameters: Sequence[float] | ArrayLike1D,
        resids: ArrayLike,
        sigma2: ArrayLike,
        individual: bool = False,
    ) -> float | Float64Array:
        r"""Computes the log-likelihood of assuming residuals are mixture normally
        distributed, conditional on the variance

        Parameters
        ----------
        parameters : ndarray
            Parameters of the first normal distribution: p1,u1,sigma1. Second one can be calculated by restrictions.
        resids  : ndarray
            The residuals to use in the log-likelihood calculation
        sigma2 : ndarray
            Conditional variances of resids
        individual : bool, optional
            Flag indicating whether to return the vector of individual log
            likelihoods (True) or the sum (False)

        Returns
        -------
        ll : float
            The log-likelihood

        Notes
        -----
        The log-likelihood of a single data point x is

        .. math::

            \ln f\left(x\right)=
            \ln \frac{1}{\sqrt{h_t}} \left[
                 p_1  \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(x-\mu_1)^2}{2\sigma_1^2} \}
                +(1-p_1)\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(x-\mu_2)^2}{2\sigma_2^2} \} \right]

        """
        parameters = asarray(parameters, dtype=float)
        p1,mu1, mu2, sigma_1, sigma_2 = parameters
        p2 = 1- p1

        z = resids/sqrt(sigma2)
        warnings.filterwarnings("ignore")
        lls =-1/2 *log(sigma2) + log(p1 * norm.pdf(z, mu1, sigma_1) + p2 * norm.pdf(z, mu2, sigma_2))
        
        warnings.filterwarnings("default")
          
        if individual:
            return lls
        else:
            return sum(lls)

    def starting_values(self, std_resid: Float64Array) -> Float64Array:
        """
        Starting values of parameters
        """
        return array([0.5,-0.2,0.2,1,1])
    
    def _simulator(self, size: int | tuple[int, ...]) -> Float64Array:
        assert self._parameters is not None
        p1,mu1, mu2, sigma_1, sigma_2 = self._parameters
        p2 = 1- p1
        
        Z = np.random.binomial(n=1, p=p1, size=size)
        return sigma_1**Z*sigma_2**(1-Z)*self._generator.standard_normal(size) + u1*Z+ u2*(1-Z)

    def simulate(
        self, parameters: int | float | Sequence[float | int] | ArrayLike1D
    ) -> Callable[[int | tuple[int, ...]], Float64Array]:
        parameters = ensure1d(parameters, "parameters", False)
        self._parameters = asarray(parameters, dtype=float)
        return self._simulator

    def parameter_names(self) -> list[str]:
        return ['p_1','mu_1','mu_2','sigma_1','sigma_1']

    def cdf(
        self,
        resids: Sequence[float] | ArrayLike1D,
        parameters: None | Sequence[float] | ArrayLike1D = None,
    ) -> Float64Array:
        self._check_constraints(parameters)
        
        parameters = asarray(parameters, dtype=float)
        p1,mu1, mu2, sigma_1, sigma_2 = self._parameters
        p2 = 1- p1
        
        return p1*stats.norm.cdf(asarray((resids-u1)/sigma_1)  ) + p2*stats.norm.cdf(asarray((resids-u2)/sigma_2))

    def ppf(
        self,
        pits: float | Sequence[float] | ArrayLike1D,
        parameters: None | Sequence[float] | ArrayLike1D = None,
    ) -> Float64Array:
        self._check_constraints(parameters)
        parameters = asarray(parameters, dtype=float)
        
        scalar = isscalar(pits)
        if scalar:
            pits = array([pits])
        else:
            pits = asarray(pits)
            
        def inverse_cdf(cdf, target_p, lower_bound=-100, upper_bound=100):
            def root_func(x):
                return cdf(x,parameters) - target_p
            return bisect(root_func, lower_bound, upper_bound)     
        
        ppf = inverse_cdf(self.cdf, pits, lower_bound=-100, upper_bound=100)

        if scalar:
            return ppf[0]
        else:
            return ppf
        
   

    def moment(
        self, n: int, parameters: None | Sequence[float] | ArrayLike1D = None
    ) -> float:
        if n < 0:
            return nan
        parameters = asarray(parameters, dtype=float)
        p1,mu1, mu2, sigma_1, sigma_2 = self._parameters
        p2 = 1- p1
        
        moment1 = stats.norm.moment(n,loc=u1,scale=sigma_1)
        moment2 = stats.norm.moment(n,loc=u2,scale=sigma_2)
        return p1 * moment1 + p2 * moment2

    def partial_moment(
        self,
        n: int,
        z: float = 0.0,
        parameters: None | Sequence[float] | ArrayLike1D = None,
        num_samples=100000,
    ) -> float:
        
        parameters = asarray(parameters, dtype=float)
        p1,mu1, mu2, sigma_1, sigma_2 = self._parameters
        p2 = 1- p1
        
        if n < 0:
            return nan
        elif n == 0:
            return cdf(z,parameters)
        elif n==1:
            return -p1*stats.norm.pdf(z,loc=u1,scale=sigma_1)  -p2*stats.norm.pdf(z,loc=u2,scale=sigma_2)
        else:
            -(z ** (n - 1)) * (p1*stats.norm.pdf(z,loc=u1,scale=sqrtsigma_1)+p2*stats.norm.pdf(z,loc=u2,scale=sqrtsigma_2)) 
            + (n - 1) * self.partial_moment(  n - 2, z, parameters  )

In [ ]:
def SPARCH(Y,dparams,X=None,mean='Zero', vol='GARCH',p=2,o=2,q=1,lags=None,cov = 'robust'):
    '''
    step 1: run a garch with student t distribution; estimate gaussian mixture parameters from garch residuals
    step 2: run a sparch using garch parameters
    '''
    def constraint1(params):
        p1,mu1, mu2, sigma1, sigma2 = params[-5:]
        return p1*mu1 + (1-p1)*mu2

    def constraint2(params):
        p1,mu1, mu2, sigma1, sigma2 = params[-5:]
        return p1*(mu1**2+sigma1**2) + (1-p1)*(mu2**2+sigma2**2)
    cons = {'type':'eq', 'fun': constraint1, 'fun': constraint2}
    
    #run GARCH with student t distribuion as banchmark
    garht = arch_model(y=Y,x=X, mean=mean, vol='GARCH',p=p, o=o,q=q,lags=lags,dist='studentst').fit(disp='off',cov_type=cov)
    #residuals = (garht.resid/garht.conditional_volatility).dropna()
    #print('Distribution Property of student t Residuals')
    #print('Kurtosis: ',kurtosis(residuals, fisher=True),'Skewness: ',skew(residuals))
    params = array(garht.params) +   0.3*array(garht.std_err)*np.random.normal(size=garht.std_err.shape)
    starting_values = np.concatenate( (np.array(params)[:-1],dparams )   )
    options={'maxiter': 1000,'constraints':cons}

    sparch = arch_model(y=Y,x=X, mean=mean, vol='GARCH',p=p, o=o,q=q,lags=lags)
    sparch.distribution = MixNormal()
    MixNormalres=sparch.fit(disp='off',cov_type=cov,options=options,starting_values=starting_values )
    return MixNormalres

In [ ]:
SPARCH(Y,dparams=[0.5,0,0,1,1],X=None,mean='Zero', vol='GARCH',p=2,o=2,q=1,lags=None,cov = 'robust')